### 1. Leer el archivo csv y hacer limpieza de datos

In [2]:
import pandas as pd

ruta_archivo = r"C:\Users\Lenovo\Desktop\escuela\IIEc\SeriesTiempo.csv"
df = pd.read_csv(ruta_archivo)

print(df.head())

        fecha          version  entropia_global  entropia_conectados  \
0  31/12/2012  quincenal_deses         0.803293             0.576352   
1  15/01/2013  quincenal_deses         0.795982             0.553620   
2  31/01/2013  quincenal_deses         0.796987             0.555817   
3  15/02/2013  quincenal_deses         0.802249             0.574103   
4  28/02/2013  quincenal_deses         0.784837             0.554118   

   inflacion_quincenal_deses  
0                   0.086451  
1                  -0.022977  
2                   0.331828  
3                   0.237716  
4                   0.110689  


In [3]:
#cambiamos las fechas por el formato datetime.
df['fecha'] = pd.to_datetime(df['fecha'], format='%d/%m/%Y')

In [4]:
#indexamos el dataframe por la columna fecha.
df = df.set_index('fecha')

In [5]:
#quitamos la columna version del dataframe.
df = df.drop(columns = ['version'])
print(df.head())

            entropia_global  entropia_conectados  inflacion_quincenal_deses
fecha                                                                      
2012-12-31         0.803293             0.576352                   0.086451
2013-01-15         0.795982             0.553620                  -0.022977
2013-01-31         0.796987             0.555817                   0.331828
2013-02-15         0.802249             0.574103                   0.237716
2013-02-28         0.784837             0.554118                   0.110689


### 2. Importamos la prueba de Causalidad de Granger

obs. se calculo el numero de rezagos optimo usando el Criterio de Informacion de Akaike (AIC)

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

### 3.1 Prueba: Entropia conectados causa inflacion quincenal desagregada

In [ ]:
resultado_1 = grangercausalitytests(df[['inflacion_quincenal_deses', 'entropia_conectados']], maxlag=6)
print(resultado_1)

In [8]:
# 2. Extraer y filtrar los p-values mayores a 0.05
significativos = []

for lag, contenido in resultado_1.items():
    diccionario_pruebas = contenido[
        0
    ]  # Contiene las 4 pruebas de ese rezago

    for nombre_prueba, tupla_valores in diccionario_pruebas.items():
        p_val = tupla_valores[1]  # La 2da entrada (índice 1) es el p-value
        estadistico = tupla_valores[0]  # El estadístico de prueba

        if p_val < 0.05:
            significativos.append({
                'rezago': lag,
                'prueba': nombre_prueba,
                'p_value': p_val,
            })

# 3. Convertir a DataFrame para visualizarlo fácilmente
df_significativos = pd.DataFrame(significativos)

# Mostrar todos los resultados significativos encontrados
print(
    f'Se encontraron {len(df_significativos)} pruebas significativas con p < 0.05:'
)
pd.set_option('display.max_rows', None)  # Mostrar todas las filas

print(df_significativos)


Se encontraron 18 pruebas significativas con p < 0.05:
    rezago        prueba   p_value
0        1     ssr_ftest  0.001099
1        1  ssr_chi2test  0.000938
2        1        lrtest  0.001033
3        1  params_ftest  0.001099
4        2     ssr_ftest  0.006158
5        2  ssr_chi2test  0.005239
6        2        lrtest  0.005690
7        2  params_ftest  0.006158
8        3     ssr_ftest  0.008035
9        3  ssr_chi2test  0.006471
10       3        lrtest  0.007187
11       3  params_ftest  0.008035
12       4     ssr_ftest  0.048617
13       4  ssr_chi2test  0.041346
14       4        lrtest  0.043991
15       4  params_ftest  0.048617
16       5  ssr_chi2test  0.044597
17       5        lrtest  0.048100


### 3.2 Prueba: Entropia global causa inflacion quincenal desagregada

In [10]:
resultado_2 = grangercausalitytests(df[['inflacion_quincenal_deses', 'entropia_global']], maxlag=6)
print(resultado_2)

{np.int64(1): ({'ssr_ftest': (np.float64(11.489475732320683), np.float64(0.0007861109668015023), np.float64(325.0), np.int64(1)), 'ssr_chi2test': (np.float64(11.595532431388259), np.float64(0.0006611044424477519), np.int64(1)), 'lrtest': (np.float64(11.395274510866784), np.float64(0.0007363116732536691), np.int64(1)), 'params_ftest': (np.float64(11.489475732320635), np.float64(0.0007861109668015221), np.float64(325.0), 1.0)}, [<statsmodels.regression.linear_model.RegressionResultsWrapper object at 0x000001FE03C77CB0>, <statsmodels.regression.linear_model.RegressionResultsWrapper object at 0x000001FE03C802F0>, array([[0., 1., 0.]])]), np.int64(2): ({'ssr_ftest': (np.float64(4.983467366221116), np.float64(0.007388018581986444), np.float64(322.0), np.int64(2)), 'ssr_chi2test': (np.float64(10.12170079971618), np.float64(0.006340165543840167), np.int64(2)), 'lrtest': (np.float64(9.968210524145206), np.float64(0.00684590057490533), np.int64(2)), 'params_ftest': (np.float64(4.9834673662211575

In [12]:
# 2. Extraer y filtrar los p-values mayores a 0.05
significativos_2 = []

for lag, contenido in resultado_2.items():
    diccionario_pruebas = contenido[
        0
    ]  # Contiene las 4 pruebas de ese rezago

    for nombre_prueba, tupla_valores in diccionario_pruebas.items():
        p_val = tupla_valores[1]  # La 2da entrada (índice 1) es el p-value
        estadistico = tupla_valores[0]  # El estadístico de prueba

        if p_val < 0.05:
            significativos_2.append({
                'rezago': lag,
                'prueba': nombre_prueba,
                'p_value': p_val,
            })

# 3. Convertir a DataFrame para visualizarlo fácilmente
df_significativos_2 = pd.DataFrame(significativos_2)

# Mostrar todos los resultados significativos encontrados
print(
    f'Se encontraron {len(df_significativos_2)} pruebas significativas con p < 0.05:'
)
pd.set_option('display.max_rows', None)  # Mostrar todas las filas

print(df_significativos_2)

Se encontraron 24 pruebas significativas con p < 0.05:
    rezago        prueba   p_value
0        1     ssr_ftest  0.000786
1        1  ssr_chi2test  0.000661
2        1        lrtest  0.000736
3        1  params_ftest  0.000786
4        2     ssr_ftest  0.007388
5        2  ssr_chi2test  0.006340
6        2        lrtest  0.006846
7        2  params_ftest  0.007388
8        3     ssr_ftest  0.007133
9        3  ssr_chi2test  0.005703
10       3        lrtest  0.006364
11       3  params_ftest  0.007133
12       4     ssr_ftest  0.032883
13       4  ssr_chi2test  0.027261
14       4        lrtest  0.029414
15       4  params_ftest  0.032883
16       5     ssr_ftest  0.034673
17       5  ssr_chi2test  0.027314
18       5        lrtest  0.030038
19       5  params_ftest  0.034673
20       6     ssr_ftest  0.046537
21       6  ssr_chi2test  0.035563
22       6        lrtest  0.039396
23       6  params_ftest  0.046537


### 3.3 Prueba: inflacion quincenal desagregada causa Entropia conectados

In [13]:
resultado_3 = grangercausalitytests(df[['entropia_conectados', 'inflacion_quincenal_deses']], maxlag=6)
print(resultado_3)

{np.int64(1): ({'ssr_ftest': (np.float64(5.880373470779367), np.float64(0.015855192061116852), np.float64(325.0), np.int64(1)), 'ssr_chi2test': (np.float64(5.934653841278869), np.float64(0.014845984502875503), np.int64(1)), 'lrtest': (np.float64(5.881603592347346), np.float64(0.015299880971052523), np.int64(1)), 'params_ftest': (np.float64(5.8803734707794915), np.float64(0.01585519206111578), np.float64(325.0), 1.0)}, [<statsmodels.regression.linear_model.RegressionResultsWrapper object at 0x000001FE03775940>, <statsmodels.regression.linear_model.RegressionResultsWrapper object at 0x000001FE03776D60>, array([[0., 1., 0.]])]), np.int64(2): ({'ssr_ftest': (np.float64(3.389379578900877), np.float64(0.034937454641708235), np.float64(322.0), np.int64(2)), 'ssr_chi2test': (np.float64(6.8840193931713465), np.float64(0.03200030974859472), np.int64(2)), 'lrtest': (np.float64(6.812559224920733), np.float64(0.033164355396209516), np.int64(2)), 'params_ftest': (np.float64(3.389379578900904), np.fl

In [14]:
# 2. Extraer y filtrar los p-values mayores a 0.05
significativos = []

for lag, contenido in resultado_3.items():
    diccionario_pruebas = contenido[
        0
    ]  # Contiene las 4 pruebas de ese rezago

    for nombre_prueba, tupla_valores in diccionario_pruebas.items():
        p_val = tupla_valores[1]  # La 2da entrada (índice 1) es el p-value
        estadistico = tupla_valores[0]  # El estadístico de prueba

        if p_val < 0.05:
            significativos.append({
                'rezago': lag,
                'prueba': nombre_prueba,
                'p_value': p_val,
            })

# 3. Convertir a DataFrame para visualizarlo fácilmente
df_significativos = pd.DataFrame(significativos)

# Mostrar todos los resultados significativos encontrados
print(
    f'Se encontraron {len(df_significativos)} pruebas significativas con p < 0.05:'
)
pd.set_option('display.max_rows', None)  # Mostrar todas las filas

print(df_significativos)

Se encontraron 12 pruebas significativas con p < 0.05:
    rezago        prueba   p_value
0        1     ssr_ftest  0.015855
1        1  ssr_chi2test  0.014846
2        1        lrtest  0.015300
3        1  params_ftest  0.015855
4        2     ssr_ftest  0.034937
5        2  ssr_chi2test  0.032000
6        2        lrtest  0.033164
7        2  params_ftest  0.034937
8        3     ssr_ftest  0.037382
9        3  ssr_chi2test  0.032825
10       3        lrtest  0.034583
11       3  params_ftest  0.037382


### 3.4 Prueba: inflacion quincenal desagregada causa Entropia global

In [ ]:
resultado_4 = grangercausalitytests(df[['entropia_global', 'inflacion_quincenal_deses']], maxlag=6)
print(resultado_4)

In [16]:
# 2. Extraer y filtrar los p-values mayores a 0.05
significativos = []

for lag, contenido in resultado_4.items():
    diccionario_pruebas = contenido[
        0
    ]  # Contiene las 4 pruebas de ese rezago

    for nombre_prueba, tupla_valores in diccionario_pruebas.items():
        p_val = tupla_valores[1]  # La 2da entrada (índice 1) es el p-value
        estadistico = tupla_valores[0]  # El estadístico de prueba

        if p_val < 0.05:
            significativos.append({
                'rezago': lag,
                'prueba': nombre_prueba,
                'p_value': p_val,
            })

# 3. Convertir a DataFrame para visualizarlo fácilmente
df_significativos = pd.DataFrame(significativos)

# Mostrar todos los resultados significativos encontrados
print(
    f'Se encontraron {len(df_significativos)} pruebas significativas con p < 0.05:'
)
pd.set_option('display.max_rows', None)  # Mostrar todas las filas

print(df_significativos)

Se encontraron 0 pruebas significativas con p < 0.05:
Empty DataFrame
Columns: []
Index: []


### Supuestos del Modelo

media de las series de tiempo = 0

varianza de las series de tiempo = c constantes

debe de haber una relacion lineal entre las entropias o indices inflacionarios pasados y los actuales.

Calculo para escoger el numero de rezagos que elimine la memoria temporal y la autocorrelacion de los residuos.

### Aspectos a mejorar en el Flujo de Trabajo

Usar un maxlag = 108 implica intentar estimar mas de 100 parametros con poco mas de 220 observaciones efectivas.

Mejora: calcular el maxlag optimo para evitar errores en el calculo.

Verificar empiricamente los supuestos de los residuos: autocorrelacion residual, heterocedasticidad condicional, normalidad de los residuos.

### Limitaciones del Modelo

1. La causalidad de granger evalua estrictamente capacida predictiva lineal en media y asimetria temporal, no una relacion causa-efecto.

Ejemplo: Los gallos cantan antes del amanecer, existe una relacion entre el canto de los gallos y el amanecer, sin embargo los gallos no provocan el amanecer.

2. Si existe un factor macroeconomico como un shock global, una recesion, cambios en tasas o tipo de cambio, afectara a la entropia. Despues, a la inflacion. La entropia parecera como causal en el sentido granger cuando no fue el caso.

3. Inconsistencias entre pruebas estadisticas: A medida que los rezagos aumentan los p-values de las pruebas ssr_ftest y params_ftest aumentan ya que sobreestriman la significancia cuando el numero de parametros a estimar es elevado respecto al tamanio muestral disponible.











